# Projet Fin de Module — Machine Learning
**Dataset :** `bienetre.csv` — 3 classes : 0 = Sain et Actif, 1 = Modéré, 2 = À Risque

**Plan :** Exploration → Normalisation → PCA (2D / 3D / 95%) → KNN (Grid Search + CV) → Régression Logistique (CV + ROC) → Arbre de Décision (Grid Search + CV) → KMeans (4 représentations)

## 1. Imports et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, silhouette_score
)
from sklearn.multiclass import OneVsRestClassifier

plt.rcParams['figure.dpi'] = 100
COLORS = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
LABELS = {0: 'Sain (0)', 1: 'Modéré (1)', 2: 'À Risque (2)'}
print('Imports OK')

## 2. Exploration du Dataset

In [ ]:
df = pd.read_csv('bienetre.csv')
print(f'Dimensions : {df.shape[0]} individus × {df.shape[1]} variables')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
missing = df.isnull().sum()
print('Valeurs manquantes par colonne :')
print(missing[missing > 0] if missing.any() else 'Aucune valeur manquante ✓')

In [ ]:
counts = df['target'].value_counts().sort_index()
labels_cls = ['Sain et Actif (0)', 'Modéré (1)', 'À Risque (2)']
colors_cls = ['#2ecc71', '#f39c12', '#e74c3c']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(labels_cls, counts.values, color=colors_cls, edgecolor='white')
ax1.set_title('Répartition des classes')
ax1.set_ylabel('Nombre d\'individus')
for i, v in enumerate(counts.values):
    ax1.text(i, v + 10, str(v), ha='center', fontweight='bold')

ax2.pie(counts.values, labels=labels_cls, colors=colors_cls,
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Distribution en pourcentage')
plt.tight_layout()
plt.show()

In [ ]:
features_to_plot = ['age', 'imc', 'cholesterol', 'activite_physique', 'stress', 'revenu']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(features_to_plot):
    for cls in [0, 1, 2]:
        data_cls = df[df['target'] == cls][feat]
        axes[i].hist(data_cls, bins=30, alpha=0.6, color=COLORS[cls], label=LABELS[cls])
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribution des variables clés par classe', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation
plt.figure(figsize=(14, 10))
corr = df.drop('target', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=False, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matrice de corrélation entre les variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Normalisation Standard

**Pourquoi ?** KNN et PCA mesurent des distances. Sans normalisation, les grandes plages (ex: revenu 8000–87000) dominent les petites (ex: IMC 17–42).

**Comment ?** `StandardScaler` : z = (x − moyenne) / écart-type → moyenne = 0, écart-type = 1

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('Avant normalisation — revenu :')
print(f'  Moyenne : {X["revenu"].mean():.2f}, Écart-type : {X["revenu"].std():.2f}')
print('Après normalisation — revenu :')
print(f'  Moyenne : {X_scaled["revenu"].mean():.4f}, Écart-type : {X_scaled["revenu"].std():.4f}')

In [ ]:
vars_viz = ['age', 'revenu', 'imc', 'cholesterol']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i, var in enumerate(vars_viz):
    axes[0, i].hist(X[var], bins=40, color='#3498db', alpha=0.7, edgecolor='white')
    axes[0, i].set_title(f'{var}\n(avant)', fontweight='bold')
    axes[1, i].hist(X_scaled[var], bins=40, color='#e74c3c', alpha=0.7, edgecolor='white')
    axes[1, i].set_title(f'{var}\n(après)', fontweight='bold')

plt.suptitle('Distributions avant / après normalisation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
X.plot(kind='box', ax=ax1, vert=False, patch_artist=True)
ax1.set_title('Avant normalisation — échelles très différentes', fontweight='bold')
X_scaled.plot(kind='box', ax=ax2, vert=False, patch_artist=True)
ax2.set_title('Après normalisation — toutes les variables à la même échelle', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. PCA — Réduction de Dimensionnalité

**Pourquoi ?** 20 dimensions = impossible à visualiser. La PCA trouve les directions où les données varient le plus.

On crée 3 représentations : **2D**, **3D**, et **95% de variance**.

In [ ]:
# Analyse complète de la variance
pca_full = PCA()
pca_full.fit(X_scaled)
variance_cumul = np.cumsum(pca_full.explained_variance_ratio_) * 100

# Nombre de composantes pour atteindre 95% de variance
n_components_95 = int(np.argmax(variance_cumul >= 95)) + 1
print(f'Composantes nécessaires pour 95% de variance : {n_components_95}')
print(f'Variance conservée avec {n_components_95} composantes : {variance_cumul[n_components_95-1]:.2f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(1, 21), pca_full.explained_variance_ratio_ * 100, color='#3498db', alpha=0.7)
ax1.set_xlabel('Composante')
ax1.set_ylabel('Variance expliquée (%)')
ax1.set_title('Variance expliquée par composante')

ax2.plot(range(1, 21), variance_cumul, 'bo-', linewidth=2)
ax2.axhline(95, color='red', linestyle='--', label='95%')
ax2.axvline(n_components_95, color='orange', linestyle='--',
            label=f'{n_components_95} composantes')
ax2.set_xlabel('Nombre de composantes')
ax2.set_ylabel('Variance cumulée (%)')
ax2.set_title('Variance cumulée')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# PCA 2D — Visualisation en 2 composantes principales
pca2 = PCA(n_components=2)
X_pca2 = pca2.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
for cls in [0, 1, 2]:
    mask = (y == cls).values
    plt.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                c=COLORS[cls], label=LABELS[cls], alpha=0.5, s=15)

plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA — Visualisation en 2D (2 composantes principales)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Variance totale conservée : {pca2.explained_variance_ratio_.sum()*100:.2f}%')

In [ ]:
# PCA 3D — Visualisation en 3 composantes principales
pca3 = PCA(n_components=3)
X_pca3 = pca3.fit_transform(X_scaled)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

for cls in [0, 1, 2]:
    mask = (y == cls).values
    ax.scatter(X_pca3[mask, 0], X_pca3[mask, 1], X_pca3[mask, 2],
               c=COLORS[cls], label=LABELS[cls], alpha=0.4, s=10)

ax.set_xlabel(f'PC1 ({pca3.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca3.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_zlabel(f'PC3 ({pca3.explained_variance_ratio_[2]*100:.1f}%)')
ax.set_title('PCA — Visualisation en 3D (3 composantes principales)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Variance totale conservée : {pca3.explained_variance_ratio_.sum()*100:.2f}%')

## 5. Préparation des 4 représentations

Chaque modèle sera évalué et cross-validé sur **4 représentations** :

| Représentation | Dimensions | Info conservée |
|---|---|---|
| Originale normalisée | 20 | 100% |
| PCA 2D | 2 | ~voir cellule PCA |
| PCA 3D | 3 | ~voir cellule PCA |
| PCA 95% variance | n_components_95 | ≥ 95% |

In [ ]:
pca95 = PCA(n_components=n_components_95)
X_pca95 = pca95.fit_transform(X_scaled)

X_arr = np.array(X_scaled)
y_arr = y.values

representations = {
    f'Originale normalisée (20 var.)': X_arr,
    'PCA 2D': X_pca2,
    'PCA 3D': X_pca3,
    f'PCA 95% ({n_components_95} composantes)': X_pca95
}

for nom, data in representations.items():
    print(f'{nom:40s} → shape: {data.shape}')

## 6. KNN — K Plus Proches Voisins

**Principe :** Pour classer un nouveau point, on regarde ses K voisins les plus proches et on vote à la majorité.

**Étape 1 :** Grid Search pour déterminer la valeur optimale de K

**Étape 2 :** Cross-validation sur les 4 représentations

In [ ]:
# Grid Search KNN — déterminer K optimal
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn,
                        cv=5, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_arr, y_arr)

print('=== Meilleurs hyperparamètres KNN ===')
print(grid_knn.best_params_)
print(f'Meilleure accuracy (CV) : {grid_knn.best_score_*100:.2f}%')

In [ ]:
# Visualisation : impact de K sur l'accuracy
results_knn = pd.DataFrame(grid_knn.cv_results_)
knn_uniform_eucl = results_knn[
    (results_knn['param_weights'] == 'uniform') &
    (results_knn['param_metric'] == 'euclidean')
].sort_values('param_n_neighbors')

best_k = grid_knn.best_params_['n_neighbors']

plt.figure(figsize=(9, 4))
plt.plot(knn_uniform_eucl['param_n_neighbors'],
         knn_uniform_eucl['mean_test_score'] * 100, 'bo-', linewidth=2)
plt.axvline(best_k, color='red', linestyle='--', label=f'K optimal = {best_k}')
plt.xlabel('Valeur de K')
plt.ylabel('Accuracy CV (%)')
plt.title('Impact de K sur l\'accuracy (weights=uniform, metric=euclidean)', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation KNN (K optimal) sur les 4 représentations
best_knn = grid_knn.best_estimator_
scores_knn = {}

print('=== KNN — Cross-validation 5 folds sur les 4 représentations ===')
for nom, data in representations.items():
    scores = cross_val_score(best_knn, data, y_arr, cv=5, scoring='accuracy')
    scores_knn[nom] = scores
    print(f'{nom:40s} → {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%')

# Visualisation
fig, ax = plt.subplots(figsize=(12, 5))
for i, (nom, scores) in enumerate(scores_knn.items()):
    ax.scatter([i]*5, scores*100, color='#3498db', alpha=0.7, zorder=3, s=60)
    ax.hlines(scores.mean()*100, i-0.3, i+0.3, colors='#2c3e50', linewidth=2.5)
    ax.text(i, scores.mean()*100 + 0.3,
            f'{scores.mean()*100:.2f}%', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(scores_knn)))
ax.set_xticklabels(list(scores_knn.keys()), rotation=15, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('KNN — Cross-validation sur les 4 représentations', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Régression Logistique

**Principe :** Le modèle calcule la probabilité d'appartenir à chaque classe (softmax pour 3 classes).

**Étape 1 :** Cross-validation sur les 4 représentations

**Étape 2 :** Courbe ROC (one-vs-rest) pour déterminer le seuil optimal

In [ ]:
# Cross-validation Régression Logistique sur les 4 représentations
lr_base = LogisticRegression(max_iter=1000, random_state=42)
scores_lr = {}

print('=== Régression Logistique — Cross-validation 5 folds ===')
for nom, data in representations.items():
    scores = cross_val_score(lr_base, data, y_arr, cv=5, scoring='accuracy')
    scores_lr[nom] = scores
    print(f'{nom:40s} → {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%')

# Visualisation
fig, ax = plt.subplots(figsize=(12, 5))
for i, (nom, scores) in enumerate(scores_lr.items()):
    ax.scatter([i]*5, scores*100, color='#2ecc71', alpha=0.7, zorder=3, s=60)
    ax.hlines(scores.mean()*100, i-0.3, i+0.3, colors='#2c3e50', linewidth=2.5)
    ax.text(i, scores.mean()*100 + 0.3,
            f'{scores.mean()*100:.2f}%', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(scores_lr)))
ax.set_xticklabels(list(scores_lr.keys()), rotation=15, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Régression Logistique — Cross-validation sur les 4 représentations', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Courbe ROC — Régression Logistique (One-vs-Rest)
# Le seuil optimal = point de la courbe le plus proche du coin supérieur gauche (0, 1)

classes = [0, 1, 2]
class_names_roc = ['Sain (0)', 'Modéré (1)', 'À Risque (2)']
roc_colors = ['#2ecc71', '#f39c12', '#e74c3c']

# Split train/test pour la ROC
X_tr, X_te, y_tr, y_te = train_test_split(
    X_arr, y_arr, test_size=0.2, random_state=42, stratify=y_arr
)
y_te_bin = label_binarize(y_te, classes=classes)

# Entraînement OneVsRest
lr_roc = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
lr_roc.fit(X_tr, y_tr)
y_score = lr_roc.predict_proba(X_te)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
seuils_optimaux = {}

for i, (cls, name, color) in enumerate(zip(classes, class_names_roc, roc_colors)):
    fpr, tpr, thresholds = roc_curve(y_te_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)

    # Seuil optimal : distance minimale au point (0, 1)
    distances = np.sqrt(fpr**2 + (1 - tpr)**2)
    idx_opt = np.argmin(distances)
    seuil_opt = thresholds[idx_opt]
    seuils_optimaux[name] = seuil_opt

    axes[i].plot(fpr, tpr, color=color, linewidth=2, label=f'AUC = {roc_auc:.3f}')
    axes[i].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[i].scatter(fpr[idx_opt], tpr[idx_opt], s=120, color='black', zorder=5,
                    label=f'Seuil optimal = {seuil_opt:.3f}')
    axes[i].set_xlabel('Taux de faux positifs')
    axes[i].set_ylabel('Taux de vrais positifs')
    axes[i].set_title(f'ROC — {name}')
    axes[i].legend(loc='lower right', fontsize=9)

plt.suptitle('Courbes ROC — Régression Logistique (One-vs-Rest)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== Seuils optimaux ===')
for name, seuil in seuils_optimaux.items():
    print(f'{name} : seuil optimal = {seuil:.4f}')
print('\nInterprétation : si P(classe X) > seuil optimal → prédire classe X')

## 8. Arbre de Décision

**Principe :** Une série de questions (si IMC > 30 ?) basées sur l'indice de Gini pour séparer les classes.

**Étape 1 :** Grid Search pour `max_depth` (profondeur maximale) et `min_samples_leaf` (taille des feuilles)

**Étape 2 :** Cross-validation sur les 4 représentations

In [ ]:
# Grid Search Arbre de Décision : max_depth + min_samples_leaf
param_grid_dt = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_leaf': [1, 2, 5, 10, 20]
}

grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt,
                       cv=5, scoring='accuracy', n_jobs=-1)
grid_dt.fit(X_arr, y_arr)

print('=== Meilleurs hyperparamètres Arbre de Décision ===')
print(f'max_depth          : {grid_dt.best_params_["max_depth"]}')
print(f'min_samples_leaf   : {grid_dt.best_params_["min_samples_leaf"]}')
print(f'Meilleure accuracy : {grid_dt.best_score_*100:.2f}%')

In [ ]:
# Visualisation : impact de max_depth et min_samples_leaf
results_dt = pd.DataFrame(grid_dt.cv_results_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Impact de max_depth (min_samples_leaf=1)
dt_depth = results_dt[results_dt['param_min_samples_leaf'] == 1].copy()
dt_depth = dt_depth.sort_values('param_max_depth')
depth_labels = [str(d) for d in dt_depth['param_max_depth']]
axes[0].bar(depth_labels, dt_depth['mean_test_score']*100, color='#e74c3c', alpha=0.7)
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Accuracy CV (%)')
axes[0].set_title('Impact de max_depth (min_samples_leaf=1)', fontweight='bold')

# Impact de min_samples_leaf (max_depth optimal)
best_depth = grid_dt.best_params_['max_depth']
dt_leaf = results_dt[results_dt['param_max_depth'] == best_depth].sort_values('param_min_samples_leaf')
axes[1].bar(dt_leaf['param_min_samples_leaf'].astype(str),
            dt_leaf['mean_test_score']*100, color='#9b59b6', alpha=0.7)
axes[1].set_xlabel('min_samples_leaf')
axes[1].set_ylabel('Accuracy CV (%)')
axes[1].set_title(f'Impact de min_samples_leaf (max_depth={best_depth})', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisation de l'arbre optimal
best_dt = grid_dt.best_estimator_
best_dt.fit(X_tr, y_tr)

plt.figure(figsize=(20, 8))
plot_tree(best_dt, max_depth=3, feature_names=X.columns.tolist(),
          class_names=['Sain', 'Modéré', 'À Risque'],
          filled=True, rounded=True, fontsize=9)
plt.title(
    f'Arbre de Décision optimal — max_depth={grid_dt.best_params_["max_depth"]}, '
    f'min_samples_leaf={grid_dt.best_params_["min_samples_leaf"]} (3 premiers niveaux)',
    fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# Importance des variables
importances = pd.Series(best_dt.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='#e74c3c', alpha=0.8)
plt.title('Importance des variables — Arbre de Décision optimal', fontweight='bold')
plt.xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation Arbre de Décision sur les 4 représentations
scores_dt = {}

print('=== Arbre de Décision — Cross-validation 5 folds sur les 4 représentations ===')
for nom, data in representations.items():
    gdt_local = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt,
                             cv=5, scoring='accuracy', n_jobs=-1)
    gdt_local.fit(data, y_arr)
    scores = cross_val_score(gdt_local.best_estimator_, data, y_arr, cv=5, scoring='accuracy')
    scores_dt[nom] = scores
    print(f'{nom:40s} → {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%  '
          f'(depth={gdt_local.best_params_["max_depth"]}, '
          f'leaf={gdt_local.best_params_["min_samples_leaf"]})')

# Visualisation
fig, ax = plt.subplots(figsize=(12, 5))
for i, (nom, scores) in enumerate(scores_dt.items()):
    ax.scatter([i]*5, scores*100, color='#e74c3c', alpha=0.7, zorder=3, s=60)
    ax.hlines(scores.mean()*100, i-0.3, i+0.3, colors='#2c3e50', linewidth=2.5)
    ax.text(i, scores.mean()*100 + 0.3,
            f'{scores.mean()*100:.2f}%', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(scores_dt)))
ax.set_xticklabels(list(scores_dt.keys()), rotation=15, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Arbre de Décision — Cross-validation sur les 4 représentations', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. KMeans — Clustering Non Supervisé

**Principe :** Regrouper les individus sans utiliser `target`. On applique KMeans sur les **4 représentations**.

**Méthode du coude :** tracer l'inertie en fonction de K → prendre le "coude"

**Score de silhouette :** mesure la qualité du clustering (-1 à 1, proche de 1 = bon)

In [ ]:
# KMeans sur les 4 représentations — Méthode du coude + Silhouette
K_range = range(1, 11)
resultats_kmeans = {}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
cluster_palette = ['#3498db', '#e67e22', '#9b59b6']

for col, (nom, data) in enumerate(representations.items()):
    # Méthode du coude
    inerties = []
    for k in K_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(data)
        inerties.append(km.inertia_)

    axes[0, col].plot(K_range, inerties, 'bo-', linewidth=2, markersize=6)
    axes[0, col].axvline(3, color='red', linestyle='--', alpha=0.7, label='K=3')
    axes[0, col].set_title(f'Coude — {nom.split(" (")[0]}', fontsize=9, fontweight='bold')
    axes[0, col].set_xlabel('K')
    axes[0, col].set_ylabel('Inertie')
    axes[0, col].legend(fontsize=8)

    # KMeans K=3
    km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
    clusters = km3.fit_predict(data)
    sil = silhouette_score(data, clusters)
    resultats_kmeans[nom] = {'clusters': clusters, 'silhouette': sil}

    # Visualisation des clusters sur PCA 2D
    viz = X_pca2 if data.shape[1] != 2 else data
    for c in [0, 1, 2]:
        mask = (clusters == c)
        axes[1, col].scatter(viz[mask, 0], viz[mask, 1],
                             c=cluster_palette[c], alpha=0.4, s=8, label=f'Cluster {c}')
    axes[1, col].set_title(f'Silhouette = {sil:.3f}', fontsize=9, fontweight='bold')
    axes[1, col].set_xlabel('PC1')
    axes[1, col].set_ylabel('PC2')
    axes[1, col].legend(fontsize=7)

plt.suptitle('KMeans — Méthode du coude et clusters sur les 4 représentations',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== Scores de silhouette (K=3) ===')
for nom, res in resultats_kmeans.items():
    print(f'{nom:40s} → silhouette = {res["silhouette"]:.4f}')

In [ ]:
# Comparaison clusters vs vraies classes (données originales normalisées)
nom_orig = list(representations.keys())[0]
clusters_orig = resultats_kmeans[nom_orig]['clusters']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for c in [0, 1, 2]:
    mask = (clusters_orig == c)
    ax1.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                c=cluster_palette[c], alpha=0.5, s=10, label=f'Cluster {c}')
ax1.set_title('KMeans — Clusters trouvés (sans connaître les classes)', fontweight='bold')
ax1.legend()

for cls in [0, 1, 2]:
    mask = (y.values == cls)
    ax2.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                c=COLORS[cls], alpha=0.5, s=10, label=LABELS[cls])
ax2.set_title('Vraies classes (target)', fontweight='bold')
ax2.legend()

for ax in [ax1, ax2]:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle('KMeans vs Vraies classes (projection PCA 2D)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Récapitulatif des Résultats

In [ ]:
print('=' * 85)
print('RÉCAPITULATIF — Accuracy moyenne (Cross-validation 5 folds)')
print('=' * 85)

all_scores = {
    'KNN': scores_knn,
    'Rég. Logistique': scores_lr,
    'Arbre Décision': scores_dt
}

header = f'{"Représentation":<42}' + ''.join(f'{m:>16}' for m in all_scores)
print(header)
print('-' * 85)

for nom in representations.keys():
    row = f'{nom:<42}'
    for modele, scores_dict in all_scores.items():
        s = scores_dict.get(nom)
        row += f'{s.mean()*100:>13.2f}%  ' if s is not None else f'{"N/A":>16}'
    print(row)

print('=' * 85)
print(f'\nKMeans — Scores de silhouette (K=3) :')
for nom, res in resultats_kmeans.items():
    print(f'  {nom:<40} silhouette = {res["silhouette"]:.4f}')

print(f'\nROC — Seuils optimaux Régression Logistique :')
for name, seuil in seuils_optimaux.items():
    print(f'  {name:<20} seuil = {seuil:.4f}')